# 03. Experiment Analysis

This notebook analyzes deterministic A/B assignments using exposure events and a session-level primary outcome. It uses Wilson confidence intervals and refuses to declare a winner from tiny samples.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,None,False
1,sessions,None,False
2,feedback,None,False
3,inventory,None,False


In [2]:
def prop(frame, key, default=None):
    return frame["properties"].map(
        lambda value: value.get(key, default)
        if isinstance(value, dict) else default
    )

if events.empty:
    print("No events export found.")
else:
    exposures = events[events["event_name"].eq("experiment_exposure")].copy()
    exposures["experiment_slug"] = prop(exposures, "experimentSlug")
    exposures["variant"] = prop(exposures, "variant")
    display(
        exposures.groupby(["experiment_slug", "variant"])
        .agg(
            exposures=("session_id", "nunique"),
            visitors=("visitor_id", "nunique"),
        )
    )

No events export found.


In [3]:
PRIMARY_EVENT = "outing_saved"
MINIMUM_PER_VARIANT = 40

if "exposures" in globals() and not exposures.empty:
    outcome_sessions = set(
        events.loc[events["event_name"].eq(PRIMARY_EVENT), "session_id"]
    )

    rows = []
    for (experiment_slug, variant), group in exposures.groupby(
        ["experiment_slug", "variant"]
    ):
        exposed_sessions = set(group["session_id"])
        conversions = len(exposed_sessions.intersection(outcome_sessions))
        total = len(exposed_sessions)
        low, high = wilson_interval(conversions, total)
        rows.append({
            "experiment": experiment_slug,
            "variant": variant,
            "exposures": total,
            "conversions": conversions,
            "conversion_rate_pct": round(100 * safe_rate(conversions, total), 1),
            "ci_low_pct": round(100 * low, 1),
            "ci_high_pct": round(100 * high, 1),
            "minimum_reached": total >= MINIMUM_PER_VARIANT,
        })

    experiment_results = pd.DataFrame(rows)
    display(experiment_results)

## Interpretation rule

A higher observed conversion rate is not automatically a winner. Review exposure balance, confidence intervals, guardrails, instrumentation quality, and whether each variant reached the planned sample.